# E1 · Limpieza, integración y Puerta de Calidad

Notebook que recorre **paso a paso** el pipeline de `src/`. Ejecútenlo completo (Kernel → Restart & Run All) y **guarden el notebook con las salidas** antes de hacer el tag `e1`.

Datos: precios de abasto (producto · calidad · presentación · origen). Una *serie* = una combinación de esas cuatro claves; todo se calcula **dentro de cada serie**.

Flujo: Bronce → esquema común → integración → **Puerta de Calidad** (cuarentena) → Plata → faltantes/atípicos → Oro.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))   # raíz del repo
import pandas as pd, matplotlib.pyplot as plt
from src import config as C, integracion as I, limpieza as L
from src.calidad import puerta_de_calidad
BASE = None   # None = data/ del repo ; "data_demo" = datos sintéticos
R = C.rutas(None if BASE is None else C.RAIZ / BASE)
pd.set_option("display.width", 160)

## 1. Carga de Bronce y esquema común
Renombrado de columnas, fecha → `datetime`, claves de serie normalizadas (sin acentos/mayúsculas), precios con `$`/comas → `float`. **N/D = faltante legítimo**; lo realmente no convertible **no se borra**: se marca con un motivo.

In [ ]:
crudas = {n: I.cargar_fuente(n, R["bronce"]) for n in C.FUENTES}
for n, d in crudas.items():
    print(n, d.shape); display(d.head(3))
est = {n: I.estandarizar_esquema(d, n) for n, d in crudas.items()}
est["A"].dtypes

## 2. Integración y resolución de conflictos
Unión por la llave (`periodo` + claves de serie). Conflictos resueltos: encabezados con acentos/puntos, formato de fecha (dd/mm vs ISO), `$` y separadores, mayúsculas/acentos y duplicados en fuentes secundarias (a cuarentena, no se descartan en silencio).

In [ ]:
integrado, cuar_integ = I.integrar(est)
print("filas integradas:", len(integrado), "| duplicados secundaria a cuarentena:", len(cuar_integ))
integrado[["periodo","serie","precio_min","precio_frec","precio_max","obs"]].head()

## 3. Puerta de Calidad (Pandera)
Esquema estricto: tipos, claves no nulas, periodo ≥ `PERIODO_MIN`, rangos de precio, llave única y coherencia `mín ≤ frec ≤ máx`. Validación *lazy* (reporta **todos** los fallos). Los inválidos se guardan en `data/cuarentena/cuarentena.csv` con `_motivo_cuarentena`.

In [ ]:
validos, cuar, motivos = puerta_de_calidad(integrado)
cuar = pd.concat([cuar, cuar_integ], ignore_index=True)
print(f"válidos: {len(validos)} | cuarentena: {len(cuar)}")
pd.Series(motivos).rename("registros").to_frame()

In [ ]:
R["cuarentena"].mkdir(parents=True, exist_ok=True)
cuar.to_csv(R["cuarentena"] / "cuarentena.csv", index=False)
cuar[["periodo","serie","precio_min","precio_frec","precio_max","_motivo_cuarentena"]].head(10)

## 4. Valores faltantes: mecanismo e imputación
Se contrasta el indicador de faltante contra las demás variables (Mann-Whitney / χ², Bonferroni). Asociación → **MAR**; sin asociación → **MCAR no rechazado**. **MNAR** no se puede probar con datos observados: se argumenta con conocimiento del sector (**[COMPLETAR: ¿"N/D" aparece justo cuando el producto escasea o el precio se dispara? Eso sería MNAR]**).

In [ ]:
diag = L.diagnosticar_faltantes(validos)
pd.DataFrame({c: {"% faltante": v["pct_faltante"], "mecanismo": v["mecanismo"]} for c, v in diag.items()}).T

In [ ]:
imputado, imp = L.imputar(validos)
print("valores imputados por variable:", imp)
fig, ax = plt.subplots(figsize=(9, 3.5))
r = imputado[imputado.serie == imputado.groupby("serie").size().idxmax()]
ax.plot(r.periodo, r[C.VARIABLE_OBJETIVO], label="serie"); 
ax.scatter(r.periodo[r[f"{C.VARIABLE_OBJETIVO}_imputado"]], r[C.VARIABLE_OBJETIVO][r[f"{C.VARIABLE_OBJETIVO}_imputado"]], c="red", label="imputado")
ax.set_title(f"Imputación por serie: {r.serie.iloc[0]}"); ax.legend(); plt.show()

**Justificación:** serie temporal por producto ⇒ interpolación lineal dentro de la serie (máx. `LIMITE_INTERPOLACION` observaciones), luego mediana de la serie. **No** se usa mediana global (mezclaría precios de productos distintos); lo que no se pueda imputar va a cuarentena. Cada imputación queda marcada en `<variable>_imputado`.

## 5. Atípicos y ruido
Tres detectores (IQR por serie, Z-score robusto con MAD sobre desviación de mediana móvil, Isolation Forest sobre precio relativo a la mediana de su serie). Se trata solo lo que detectan **≥2 de 3** (winsorización a los límites IQR); se conserva el valor original.

In [ ]:
flag = L.detectar_atipicos(imputado)
limpio, atip = L.tratar_atipicos(flag)
display(pd.DataFrame(atip).T)
y = C.VARIABLE_OBJETIVO
med = limpio.groupby("serie")[f"{y}_original"].transform("median")
limpio["rel_antes"], limpio["rel_despues"] = limpio[f"{y}_original"] / med, limpio[y] / med
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
limpio.boxplot(column="rel_antes", by="producto", ax=ax[0]); ax[0].set_title("Precio relativo ANTES")
limpio.boxplot(column="rel_despues", by="producto", ax=ax[1]); ax[1].set_title("DESPUÉS"); plt.suptitle(""); plt.show()

## 6. Plata → Normalización, discretización y jerarquías (Oro)

In [ ]:
plata_cols = [c for c in limpio.columns if not any(s in c for s in ["_lim_inf","_lim_sup","_atipico_iqr","_atipico_zrob","_atipico_if"])]
plata = limpio[plata_cols]
oro = L.discretizar(L.normalizar(plata))
oro[[y, f"{y}_mm", f"{y}_z", f"{y}_rel", f"{y}_nivel", f"{y}_rango", "trimestre", "semestre", "macro_region"]].head()

Jerarquías: **fecha < mes < trimestre < semestre < año** y **origen (estado) < macro-región**. Discretización del precio *relativo a la mediana de su serie*: por frecuencia (terciles bajo/medio/alto) y por ancho (4 rangos). Min-Max y Z-Score se calculan **dentro de cada serie**.

## 7. Reducción de datos

In [ ]:
trim = L.agregar_periodo(oro)
keep, info = L.seleccionar_variables(oro)
muestra = L.muestra_estratificada(oro, 0.3)
print(f"Agregación observaciones→{C.FRECUENCIA_OBJETIVO}:", oro.shape, "→", trim.shape)
print("Muestreo estratificado 30%:", muestra.shape)
print("Selección de variables:", keep, "| descartadas:", info)

## 8. Guardado de capas y resumen
La ejecución oficial/automática la hace `python -m src.pipeline` (GitHub Actions).

In [ ]:
from src.pipeline import ejecutar
rep = ejecutar(None if BASE is None else C.RAIZ / BASE)
print("Series:", rep["series"], "| periodos:", rep["periodos_distintos"], "| cumple ≥36:", rep["cumple_min_periodos"], "| series con ≥36:", rep["series_con_min_periodos"])
print("Plata/Oro/cuarentena escritos en:", R["plata"].parent)

## Conclusiones
**[COMPLETAR por el equipo]** — qué defectos de datos aparecieron, cuántos registros fueron a cuarentena y por qué, y cómo cambió la distribución de la variable objetivo tras la limpieza.